In [1]:
# ===========================================================================
# UA-SPEECH DATA PIPELINE - interactive driver
#
# All logic lives in the src/ package; this notebook only calls it, so the
# notebook and run_pipeline.py cannot drift apart. To change behaviour, edit
# the module, not this notebook.
#
# EDA lives in notebooks/02_feature_analysis.ipynb, not here - this notebook
# builds the manifest (scan, verify, filter, label, split, dataset) and
# investigates the VAD/padding root cause behind the MFCC "long silent tail"
# supervisor feedback (Stage 9).
#
#   src/config.py         paths, speaker ground truth, label maps, hyperparams
#   src/extraction.py     .tgz archive extraction
#   src/scanning.py       filename parsing, verification, mic filter, labels
#   src/splits.py         LOSO (detection) and balanced 81-fold (severity)
#   src/preprocessing.py  resample, Silero VAD trim, pad, MFCC
#   src/vad.py             Silero VAD wrapper (leading/trailing trim, fallback, stats)
#   src/dataset.py        UASpeechDataset
#   src/models/           deep / acoustic / fusion pathways
# ===========================================================================

# STAGE 0 - Setup. Put the project root on the import path; autoreload picks up
# edits to src/ without a kernel restart.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.console import print_header, print_kv

config.ensure_directories()

print_header("UA-Speech Dysarthria Pipeline")
print_kv("Project root", config.PROJECT_ROOT)
print_kv("Archive folder", config.ARCHIVE_DIR)
print_kv("Audio folder", config.AUDIO_DIR)
print_kv("Ground truth", f"{len(config.CONTROL_IDS)} controls + "
                        f"{len(config.DYSARTHRIC_IDS)} dysarthric = "
                        f"{len(config.ALL_SPEAKERS)} speakers")
print_kv("Microphone channel", config.TARGET_MIC)
print_kv("Words per speaker", config.WORDS_PER_SPEAKER)


══════════════════════════════════════════════════════════════════════════════
  UA-SPEECH DYSARTHRIA PIPELINE
══════════════════════════════════════════════════════════════════════════════
  Project root ............................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification
  Archive folder .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\archives
  Audio folder ............................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\extracted
  Ground truth ............................ 13 controls + 15 dysarthric = 28 speakers
  Microphone channel ...................... M6
  Words per speaker ....................... 765


In [2]:
# STAGE 1 - Extract the archives.
# Copy UASpeech_normalized_C.tgz and UASpeech_normalized_FM.tgz into
# data/archives/ first. Runs once; the extracted audio persists in
# data/extracted/, so skip this cell on later passes.
from src.extraction import extract_all_archives

extract_all_archives()


══════════════════════════════════════════════════════════════════════════════
  DATASET EXTRACTION
══════════════════════════════════════════════════════════════════════════════
  Archive folder .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\archives
  Extract folder .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\extracted
  Extracting .............................. UASpeech_normalized_C.tgz
  [ ✓ ] Extracted UASpeech_normalized_C.tgz
  Extracting .............................. UASpeech_normalized_FM.tgz
  [ ✓ ] Extracted UASpeech_normalized_FM.tgz
  Archives extracted ...................... 2 / 2


2

In [3]:
# STAGE 2 - Scan and verify against the 28-speaker ground truth.
# Filenames parse as <Speaker>_<Block>_<WordCode>_<Mic>.wav. Two rules baked
# into parse_filename, both of which the original exploratory scan got wrong:
#   1. The mic channel is taken POSITIONALLY from the final token. The old
#      logic searched for the first token starting with 'M', which matched male
#      speaker IDs like M01 before ever reaching the real mic token.
#   2. macOS resource-fork duplicates ('._' prefix) are skipped - the archive
#      holds one per real .wav, which doubled the apparent file count.
from src.scanning import scan_audio_files, verify_speakers

df_audio = scan_audio_files()
speakers_ok = verify_speakers(df_audio)

df_audio.head()


══════════════════════════════════════════════════════════════════════════════
  DATASET SCAN
══════════════════════════════════════════════════════════════════════════════
  Scanning folder ......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\data\extracted
  Valid audio files ....................... 143565
  Skipped files ........................... 143565
  Unique speakers ......................... 28

─── Speaker Verification ─────────────────────────────────────────────────────
  Expected speakers ....................... 28
  Found speakers .......................... 28
  Missing ................................. None
  Spurious ................................ None
  [ ✓ ] All 28 ground-truth speakers present, no spurious IDs


,Filename,Speaker_ID,Group,Block,WordCode,Microphone_Channel,Filepath
0,CF02_B1_C10_M2.wav,CF02,Healthy Control,B1,C10,M2,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
1,CF02_B1_C10_M3.wav,CF02,Healthy Control,B1,C10,M3,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
2,CF02_B1_C10_M4.wav,CF02,Healthy Control,B1,C10,M4,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
3,CF02_B1_C10_M5.wav,CF02,Healthy Control,B1,C10,M5,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...
4,CF02_B1_C10_M6.wav,CF02,Healthy Control,B1,C10,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...


In [4]:
# STAGE 3 - Filter to microphone channel M6 (base-paper protocol: M6 only, all
# three blocks, all word categories, no word-type filtering).
# Expect 21,420 utterances: 11,475 dysarthric + 9,945 healthy control.
from src.scanning import filter_mic_channel

df_m6 = filter_mic_channel(df_audio)


─── Microphone Filter (M6 only) ──────────────────────────────────────────────
  Total M6 samples ........................ 21420
    Dysarthric Patient .................... 11475
    Healthy Control ....................... 9945
  Missing dysarthric speakers on M6 ....... None
  Missing control speakers on M6 .......... None


In [5]:
# STAGE 4 - Validate WAV headers. A handful of UA-Speech files extracted
# zero-filled (see README "Data verification note") - torchaudio/libsndfile
# can't read them, and they would otherwise crash training partway through a
# LOSO run. Dropped here, before word counts, so the count check reflects
# genuinely usable data.
from src.scanning import validate_wav_headers

df_m6 = validate_wav_headers(df_m6)


─── WAV Header Validation ────────────────────────────────────────────────────
  Files checked ........................... 21420
  Corrupted (dropped) ..................... 39
  Speaker_ID
  F03    24
  F04    15
  [ ✗ ] 39 corrupted file(s) excluded from the manifest - see the dropped rows' Speaker_ID/WordCode above


In [6]:
# STAGE 5 - Per-speaker word counts; every speaker should have all 765 words on
# M6. Speakers below that are flagged, NOT dropped: an incomplete speaker is
# still usable, and silently removing one would change the LOSO fold count.
from src.scanning import check_word_counts

word_counts = check_word_counts(df_m6)


─── Per-Speaker Word Counts (target: 765) ────────────────────────────────────
  Speaker_ID
  F03     741
  F04     750
  CF02    765
  CF03    765
  CM01    765
  CM04    765
  CF04    765
  CF05    765
  CM06    765
  CM05    765
  CM08    765
  CM09    765
  CM12    765
  CM10    765
  F02     765
  CM13    765
  F05     765
  M01     765
  M04     765
  M05     765
  M07     765
  M08     765
  M09     765
  M10     765
  M11     765
  M12     765
  M14     765
  M16     765
  [ ✗ ] 2 speaker(s) below 765 words (flag, don't drop)
  Speaker_ID
  F03    741
  F04    750


In [7]:
# STAGE 6 - Severity labels: four classes (Very Low / Low / Mid / High) for the
# 15 dysarthric speakers; controls get 'N/A (Control)'.
from src.scanning import add_severity_labels

df_m6 = add_severity_labels(df_m6)

df_m6.head()


─── Severity Labels ──────────────────────────────────────────────────────────
  [ ✓ ] No unmapped dysarthric speakers
  Severity
  N/A (Control)    9945
  High             3825
  Very Low         3036
  Low              2295
  Mid              2280


,Filename,Speaker_ID,Group,Block,WordCode,Microphone_Channel,Filepath,Severity
4,CF02_B1_C10_M6.wav,CF02,Healthy Control,B1,C10,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
11,CF02_B1_C11_M6.wav,CF02,Healthy Control,B1,C11,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
18,CF02_B1_C12_M6.wav,CF02,Healthy Control,B1,C12,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
25,CF02_B1_C13_M6.wav,CF02,Healthy Control,B1,C13,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)
32,CF02_B1_C14_M6.wav,CF02,Healthy Control,B1,C14,M6,C:\Users\surya\OneDrive\Desktop\Acoustic-Aware...,N/A (Control)


In [8]:
# STAGE 7 - Cross-validation splits.
#   Detection: Leave-One-Speaker-Out over all 28 speakers -> 28 folds.
#   Severity:  classes are unbalanced (4/3/3/5 speakers), so
#              config.DROPPED_FOR_BALANCE excludes M12, M08, F05 to reach 3 per
#              class -> 3^4 = 81 leave-one-per-class-out iterations.
#
# NOTE: the base paper gives no explicit exclusion list, so that set of three
# speakers is OUR ASSUMPTION. Confirm with the team before treating any
# severity result as final.
from src.splits import build_severity_folds, get_loso_split, summarize_detection_splits

summarize_detection_splits(df_m6)
severity_folds = build_severity_folds(df_m6)


══════════════════════════════════════════════════════════════════════════════
  DETECTION SPLITS (LEAVE-ONE-SPEAKER-OUT)
══════════════════════════════════════════════════════════════════════════════
  Total LOSO folds ........................ 28

─── Example fold (CF02 held out) ─────────────────────────────────────────────
  Train samples ........................... 20616
  Test samples ............................ 765

══════════════════════════════════════════════════════════════════════════════
  SEVERITY SPLITS (BALANCED LEAVE-ONE-PER-CLASS-OUT)
══════════════════════════════════════════════════════════════════════════════
  Speakers dropped for balance ............ ['M12', 'M08', 'M09']

─── Speakers per severity class ──────────────────────────────────────────────
  High .................................... F05, M10, M14
  Low ..................................... F02, M07, M16
  Mid ..................................... F04, M05, M11
  Very Low ..............................

In [9]:
# STAGE 8 - Build the PyTorch dataset. UASpeechDataset returns the raw waveform
# (Deep Pathway), the 39-dim MFCC tensor (Acoustic Pathway), both labels, and
# the speaker ID - so both pathways train on identical audio and splits.
from src.dataset import UASpeechDataset

dataset = UASpeechDataset(df_m6)
sample = dataset[0]

print_header("Dataset Build")
print_kv("Total samples", len(dataset))
print_kv("Waveform shape", tuple(sample["waveform"].shape))
print_kv("MFCC shape", tuple(sample["mfcc"].shape))
print_kv("Detection label", sample["group_label"].item())
print_kv("Severity label", sample["severity_label"].item())
print_kv("Speaker ID", sample["speaker_id"])

manifest_path = config.OUTPUT_DIR / "m6_manifest.csv"
df_m6.to_csv(manifest_path, index=False)
print_kv("Manifest saved", manifest_path)


══════════════════════════════════════════════════════════════════════════════
  DATASET BUILD
══════════════════════════════════════════════════════════════════════════════
  Total samples ........................... 21381
  Waveform shape .......................... (1, 64000)
  MFCC shape .............................. (1, 39, 401)
  Detection label ......................... 0
  Severity label .......................... -1
  Speaker ID .............................. CF02
  Manifest saved .......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv


In [10]:
# STAGE 9 - Root-cause investigation: why does the MFCC plot show a large
# near-constant region after the speech?
#
# Two candidate causes, measured here rather than assumed:
#   1. torchaudio.functional.vad() (the OLD preprocessing step, now replaced by
#      Silero VAD in src/vad.py) only trims LEADING silence - it is a forward
#      energy-ramp detector with no concept of a trailing edge. Trailing
#      silence in the raw recording would survive it untouched.
#   2. Every clip is pad/truncated to a FIXED config.CLIP_SECONDS=4.0s window
#      (config.MAX_SAMPLES=64000 samples / 400 MFCC frames). UA-Speech
#      utterances are single isolated words - if the actual spoken content is
#      much shorter than 4s, most of the window is padding regardless of how
#      good the VAD trim is.
#
# This cell measures both directly on a spread of real files: raw duration,
# resampled duration (should be identical - resampling doesn't crop), VAD
# speech duration/ratio, and what fraction of the fixed 4s window is padding.
import pandas as pd
import torchaudio

from src.preprocessing import _load_resampled, load_and_preprocess_with_stats

sample_files = (df_m6.groupby("Speaker_ID", group_keys=False)
                .apply(lambda g: g.sample(min(3, len(g)), random_state=config.DEFAULT_SEED)))
sample_files = sample_files.sample(min(80, len(sample_files)), random_state=config.DEFAULT_SEED)

records = []
for row in sample_files.itertuples(index=False):
    raw_waveform, raw_sr = torchaudio.load(row.Filepath)
    raw_duration_s = raw_waveform.shape[1] / raw_sr

    resampled_waveform, resampled_sr = _load_resampled(row.Filepath)
    resampled_duration_s = resampled_waveform.shape[1] / resampled_sr

    _, _, vad_stats = load_and_preprocess_with_stats(row.Filepath)

    records.append({
        "Filename": row.Filename, "Speaker_ID": row.Speaker_ID, "Group": row.Group,
        "raw_duration_s": raw_duration_s, "resampled_duration_s": resampled_duration_s,
        **vad_stats,
    })

root_cause_df = pd.DataFrame(records)
max_window_s = config.MAX_SAMPLES / config.TARGET_SR
root_cause_df["padding_fraction_of_window"] = (
    1 - root_cause_df["speech_duration_s"] / max_window_s).clip(lower=0)

print_header("Root-cause investigation — MFCC trailing padding")
print_kv("Sample size", len(root_cause_df))
print_kv("Fixed analysis window", f"{max_window_s:.1f}s ({config.MAX_SAMPLES} samples)")
print_kv("Raw duration (median / mean)",
        f"{root_cause_df['raw_duration_s'].median():.3f}s / {root_cause_df['raw_duration_s'].mean():.3f}s")
print_kv("Resampled duration == raw duration",
        bool((root_cause_df['raw_duration_s'] - root_cause_df['resampled_duration_s']).abs().max() < 1e-6))
print_kv("VAD speech duration (median / mean)",
        f"{root_cause_df['speech_duration_s'].median():.3f}s / {root_cause_df['speech_duration_s'].mean():.3f}s")
print_kv("VAD fallback rate", f"{root_cause_df['fallback_used'].mean():.1%}")
print_kv("Median padding fraction of the 4s window",
        f"{root_cause_df['padding_fraction_of_window'].median():.1%}")

root_cause_path = config.METRICS_DIR / "vad_root_cause_summary.csv"
root_cause_df.to_csv(root_cause_path, index=False)
print_kv("Root-cause summary saved", root_cause_path)

root_cause_df[["Filename", "Speaker_ID", "raw_duration_s", "speech_duration_s",
              "speech_ratio", "padding_fraction_of_window", "fallback_used"]].head(10)

C:\Users\surya\AppData\Local\Temp\ipykernel_15184\2528089302.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(3, len(g)), random_state=config.DEFAULT_SEED)))



══════════════════════════════════════════════════════════════════════════════
  ROOT-CAUSE INVESTIGATION — MFCC TRAILING PADDING
══════════════════════════════════════════════════════════════════════════════
  Sample size ............................. 80
  Fixed analysis window ................... 4.0s (64000 samples)
  Raw duration (median / mean) ............ 1.952s / 2.189s
  Resampled duration == raw duration ...... True
  VAD speech duration (median / mean) ..... 0.540s / 0.584s
  VAD fallback rate ....................... 0.0%
  Median padding fraction of the 4s window . 86.5%
  Root-cause summary saved ................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\vad_root_cause_summary.csv


,Filename,Speaker_ID,raw_duration_s,speech_duration_s,speech_ratio,padding_fraction_of_window,fallback_used
0,M11_B2_C14_M6.wav,M11,4.261812,0.764,0.179266,0.809,False
1,CF02_B2_CW85_M6.wav,CF02,1.877875,0.604,0.321640,0.849,False
2,M05_B2_C14_M6.wav,M05,2.114375,0.380,0.179722,0.905,False
3,CM06_B2_C14_M6.wav,CM06,1.432125,0.412,0.287684,0.897,False
4,CM01_B2_CW85_M6.wav,CM01,1.613125,0.444,0.275242,0.889,False
5,M10_B2_C14_M6.wav,M10,2.027813,0.380,0.187394,0.905,False
6,CF05_B2_C14_M6.wav,CF05,1.492312,0.540,0.361855,0.865,False
7,CM05_B2_CW85_M6.wav,CM05,1.639250,0.412,0.251334,0.897,False
8,CF03_B2_C14_M6.wav,CF03,1.495750,0.508,0.339629,0.873,False
9,M08_B3_UW88_M6.wav,M08,2.218688,0.828,0.373194,0.793,False


In [ ]:
# STAGE 10 - Is config.MAX_SAMPLES=64000 (400 MFCC frames) actually justified?
#
# Stage 9 measured this on an 80-file sample; here it's the full M6 manifest
# (reusing outputs/vad_stats.csv if Stage 9 / src.preprocessing.compute_vad_stats_batch
# already built it - this does not re-run VAD if so). Converts each
# utterance's VAD speech_duration_s into a valid MFCC frame count via
# src.preprocessing.mfcc_frame_count (the same formula the dataset, the
# Acoustic Pathway, and notebooks/02's VAD-validation figure all now share),
# then reports the distribution so max_frames=400 is a measured choice, not
# an assumption - see the module docstring in notebooks/02_feature_analysis.ipynb
# Stage 2 for how the 400-frame fixed window shows up as padding.
import numpy as np

from src.preprocessing import compute_vad_stats_batch, mfcc_frame_count

vad_stats_full = compute_vad_stats_batch(df_m6)
valid_frames_full = vad_stats_full["speech_duration_s"].apply(
    lambda s: mfcc_frame_count(int(round(s * config.TARGET_SR))))

percentiles = [0, 50, 75, 90, 95, 99, 100]
frame_distribution = {
    f"p{p}" if p not in (0, 100) else ("min" if p == 0 else "max"):
        int(np.percentile(valid_frames_full, p))
    for p in percentiles
}
frame_distribution["mean"] = float(valid_frames_full.mean())

current_max_frames = mfcc_frame_count(config.MAX_SAMPLES)

print_header("Valid MFCC frame distribution (full M6 manifest)")
print_kv("Utterances", len(valid_frames_full))
print_kv("Minimum valid frames", frame_distribution["min"])
print_kv("Median valid frames (p50)", frame_distribution["p50"])
print_kv("Mean valid frames", f"{frame_distribution['mean']:.1f}")
print_kv("75th percentile", frame_distribution["p75"])
print_kv("90th percentile", frame_distribution["p90"])
print_kv("95th percentile", frame_distribution["p95"])
print_kv("99th percentile", frame_distribution["p99"])
print_kv("Maximum valid frames", frame_distribution["max"])
print_kv("Current config.MAX_SAMPLES frame count", current_max_frames)
print_kv("Frames at/above p99 wasted as pure padding",
        f"{current_max_frames - frame_distribution['p99']} "
        f"({(current_max_frames - frame_distribution['p99']) / current_max_frames:.1%} of the fixed window)")

frame_distribution_path = config.METRICS_DIR / "mfcc_valid_frame_distribution.csv"
pd.Series(frame_distribution).to_frame("frames").to_csv(frame_distribution_path)
print_kv("Distribution saved", frame_distribution_path)

# NOTE: this is a REPORT, not an automatic change. config.MAX_SAMPLES stays at
# 4.0s / 400 frames here - shrinking it is a real architectural decision
# (every cached MFCC tensor, every checkpoint's Conv1d/attention shapes, and
# every already-run experiment's frame budget assumes it) that the team should
# make deliberately from the numbers above, not have silently changed by this
# investigation. If p99 is far below 400, a smaller max_frames (e.g. rounding
# p99 up to the nearest pooling-friendly multiple of 4, since AcousticPathway's
# two MaxPool1d(2) stages want an even frame count at each stage) would cut
# the median padding fraction dramatically without truncating almost any
# utterance's real speech.